# Testing Utility-Tuned Models

Compare baseline models vs. utility-tuned versions to see improvement from threshold optimization.

This notebook:
1. Trains baseline LogisticGLM
2. Trains utility-tuned LogisticGLM (with optimal threshold)
3. Compares utility scores
4. Shows the threshold difference


In [ ]:
import sys, logging, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add rebuild directory to imports
sys.path.insert(0, '.')
logging.basicConfig(level=logging.WARNING)

print("Importing modules...")

from data_loader import (
    load_physionet_files,
    add_hours_until_sepsis,
    split_patients_by_status,
    get_rows_for_patients,
)
from bootstrap import BootstrapResampler
from models import LogisticGLM
from models_utility_tuned import LogisticGLMUtilityTuned
from training import BootstrapEvaluator, extract_Xy
from utility import physionet_utility, find_optimal_threshold

print("✓ All imports successful")

## 1. Configuration & Data Loading

In [ ]:
# Configuration
TRAIN_SIZES = [50, 100]
BOOTSTRAP_SIZES = [25]
N_ITER = 2
RANDOM_STATE = 42

print(f"Config: {len(TRAIN_SIZES)} train × {len(BOOTSTRAP_SIZES)} boot × {N_ITER} iter = {len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER} evaluations")

print("\nLoading data...")
DATA_DIR = Path('../data/physionet_sepsis')

raw_df = load_physionet_files(DATA_DIR)
print(f"✓ Loaded: {raw_df['patient_id'].nunique():,} patients, {len(raw_df):,} rows")

print("Computing hours_until_sepsis...")
df = add_hours_until_sepsis(raw_df, keep_post_onset=True)
n_septic = df['hours_until_sepsis'].notna().sum()
print(f"✓ Computed: {n_septic:,} septic rows")

## 2. Single Configuration: Baseline vs. Utility-Tuned

In [ ]:
print("\n" + "="*70)
print("COMPARISON: BASELINE vs. UTILITY-TUNED")
print("="*70)

train_size = 50
boot_size = 25

print(f"\n[Setup] Train size: {train_size}, Bootstrap size: {boot_size}")

# Split patients
train_pids, boot_pids = split_patients_by_status(
    df, n_train_patients=train_size,
    random_state=RANDOM_STATE, stratify_by_sepsis=True
)
train_df = get_rows_for_patients(df, train_pids)

print(f"✓ Training data: {len(train_df)} rows")

# Create bootstrap resampler (shared for both models)
resampler = BootstrapResampler(
    bootstrap_pool_patient_ids=boot_pids,
    full_df=df,
    n_iterations=N_ITER,
    bootstrap_sample_size=boot_size,
    random_state=RANDOM_STATE
)

# Generate first bootstrap sample (used for both models)
_, boot_df_0 = resampler.generate_iteration(0)
print(f"✓ Bootstrap sample 0: {len(boot_df_0)} rows")


### Baseline: LogisticGLM with default threshold (0.5)

In [ ]:
print("\n" + "-"*70)
print("BASELINE: LogisticGLM with threshold=0.5")
print("-"*70)

# Baseline model
baseline_model = LogisticGLM(C=0.01)
baseline_evaluator = BootstrapEvaluator(
    model=baseline_model,
    train_df=train_df,
    label_column='SepsisLabel',
    patient_id_column='patient_id'
)

baseline_metrics = baseline_evaluator.evaluate_iteration(
    boot_df_0, 0,
    compute_per_group=False
)

print(f"\nBaseline Results (threshold=0.5):")
print(f"  Utility:  {baseline_metrics.get('utility', np.nan):.4f}")
print(f"  AUROC:    {baseline_metrics.get('auroc', np.nan):.4f}")
print(f"  Recall:   {baseline_metrics.get('recall', np.nan):.4f}")
print(f"  Accuracy: {baseline_metrics.get('accuracy', np.nan):.4f}")

### Utility-Tuned: LogisticGLMUtilityTuned (finds optimal threshold)

In [ ]:
print("\n" + "-"*70)
print("UTILITY-TUNED: LogisticGLMUtilityTuned (optimal threshold)")
print("-"*70)

# Utility-tuned model
tuned_model = LogisticGLMUtilityTuned(C=0.1)  # Note: C=0.1 instead of 0.01
tuned_evaluator = BootstrapEvaluator(
    model=tuned_model,
    train_df=train_df,
    label_column='SepsisLabel',
    patient_id_column='patient_id'
)

tuned_metrics = tuned_evaluator.evaluate_iteration(
    boot_df_0, 0,
    compute_per_group=False
)

print(f"\nUtility-Tuned Results (threshold={tuned_model.optimal_threshold:.4f}):")
print(f"  Utility:  {tuned_metrics.get('utility', np.nan):.4f}")
print(f"  AUROC:    {tuned_metrics.get('auroc', np.nan):.4f}")
print(f"  Recall:   {tuned_metrics.get('recall', np.nan):.4f}")
print(f"  Accuracy: {tuned_metrics.get('accuracy', np.nan):.4f}")

### Comparison

In [ ]:
print("\n" + "="*70)
print("COMPARISON")
print("="*70)

baseline_utility = baseline_metrics.get('utility', np.nan)
tuned_utility = tuned_metrics.get('utility', np.nan)
improvement = ((tuned_utility - baseline_utility) / abs(baseline_utility)) * 100 if baseline_utility != 0 else np.nan

comparison_df = pd.DataFrame({
    'Metric': ['Utility', 'AUROC', 'Recall', 'Accuracy'],
    'Baseline (0.5)': [
        baseline_metrics.get('utility', np.nan),
        baseline_metrics.get('auroc', np.nan),
        baseline_metrics.get('recall', np.nan),
        baseline_metrics.get('accuracy', np.nan),
    ],
    f'Tuned ({tuned_model.optimal_threshold:.3f})': [
        tuned_metrics.get('utility', np.nan),
        tuned_metrics.get('auroc', np.nan),
        tuned_metrics.get('recall', np.nan),
        tuned_metrics.get('accuracy', np.nan),
    ]
})

print("\n" + comparison_df.round(4).to_string(index=False))

print(f"\n\n✓ IMPROVEMENT: {improvement:.1f}% increase in utility")
print(f"  Baseline:    {baseline_utility:.4f}")
print(f"  Tuned:       {tuned_utility:.4f}")
print(f"  Gain:        {tuned_utility - baseline_utility:.4f}")
print(f"  Threshold:   0.5000 → {tuned_model.optimal_threshold:.4f}")

## 3. Threshold Analysis

In [ ]:
# Show why threshold tuning matters
print("\n" + "="*70)
print("THRESHOLD SENSITIVITY")
print("="*70)

# Get probabilities from baseline model
X_eval, y_eval = extract_Xy(boot_df_0, label_column='SepsisLabel')
y_proba = baseline_model.predict_proba(X_eval)[:, 1]

# Test different thresholds
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
threshold_results = []

for thresh in thresholds_to_test:
    y_pred = (y_proba >= thresh).astype(int)
    utility = physionet_utility(
        y_eval, y_pred, 
        patient_ids=boot_df_0['patient_id'].values
    )
    threshold_results.append({
        'threshold': thresh,
        'utility': utility,
        'n_positives': y_pred.sum(),
        'sensitivity': (y_pred[y_eval == 1].sum() / (y_eval == 1).sum()) if (y_eval == 1).sum() > 0 else 0,
    })

threshold_df = pd.DataFrame(threshold_results)
print("\nUtility at different thresholds:")
print(threshold_df.round(4).to_string(index=False))

optimal_idx = threshold_df['utility'].idxmax()
optimal_row = threshold_df.loc[optimal_idx]
print(f"\n✓ Optimal threshold: {optimal_row['threshold']:.1f} with utility {optimal_row['utility']:.4f}")

## 4. Full Comparison Across Configurations

In [ ]:
print("\n" + "="*70)
print("FULL COMPARISON: All Configurations")
print("="*70)

results = []
config_num = 0
total_configs = len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER

for train_size in TRAIN_SIZES:
    print(f"\nTrain size: {train_size}")
    
    # Split patients
    train_pids, boot_pids = split_patients_by_status(
        df, n_train_patients=train_size,
        random_state=RANDOM_STATE, stratify_by_sepsis=True
    )
    train_df = get_rows_for_patients(df, train_pids)
    
    for boot_size in BOOTSTRAP_SIZES:
        print(f"  Boot size: {boot_size}")
        
        # Create bootstrap resampler
        resampler = BootstrapResampler(
            bootstrap_pool_patient_ids=boot_pids,
            full_df=df,
            n_iterations=N_ITER,
            bootstrap_sample_size=boot_size,
            random_state=RANDOM_STATE
        )
        
        for iter_idx in range(N_ITER):
            config_num += 1
            print(f"    [{config_num:2d}/{total_configs}] Iteration {iter_idx+1}... ", end='', flush=True)
            
            try:
                _, boot_df = resampler.generate_iteration(iter_idx)
                
                # Baseline
                baseline_model = LogisticGLM(C=0.01)
                baseline_evaluator = BootstrapEvaluator(
                    model=baseline_model, train_df=train_df,
                    label_column='SepsisLabel', patient_id_column='patient_id'
                )
                baseline_m = baseline_evaluator.evaluate_iteration(boot_df, iter_idx)
                
                # Utility-tuned
                tuned_model = LogisticGLMUtilityTuned(C=0.1)
                tuned_evaluator = BootstrapEvaluator(
                    model=tuned_model, train_df=train_df,
                    label_column='SepsisLabel', patient_id_column='patient_id'
                )
                tuned_m = tuned_evaluator.evaluate_iteration(boot_df, iter_idx)
                
                results.append({
                    'train_size': train_size,
                    'boot_size': boot_size,
                    'iteration': iter_idx,
                    'baseline_utility': baseline_m.get('utility', np.nan),
                    'tuned_utility': tuned_m.get('utility', np.nan),
                    'optimal_threshold': tuned_model.optimal_threshold,
                })
                
                improvement = ((tuned_m['utility'] - baseline_m['utility']) / abs(baseline_m['utility'])) * 100 if baseline_m['utility'] != 0 else np.nan
                print(f"✓ {improvement:+.1f}%")
                
            except Exception as e:
                print(f"✗ ERROR: {str(e)[:40]}")
                results.append({
                    'train_size': train_size,
                    'boot_size': boot_size,
                    'iteration': iter_idx,
                    'baseline_utility': np.nan,
                    'tuned_utility': np.nan,
                    'optimal_threshold': np.nan,
                })

print(f"\n{'='*70}")
print(f"COMPLETE: {len(results)} results")
print(f"{'='*70}")

## 5. Results Summary

In [ ]:
results_df = pd.DataFrame(results)

print("\nResults by Configuration:")
print(results_df.round(4).to_string(index=False))

# Compute improvements
results_df['improvement_pct'] = (
    (results_df['tuned_utility'] - results_df['baseline_utility']) / 
    results_df['baseline_utility'].abs()
) * 100

print("\n\nImprovement Summary:")
print(f"  Mean improvement:  {results_df['improvement_pct'].mean():+.1f}%")
print(f"  Median improvement: {results_df['improvement_pct'].median():+.1f}%")
print(f"  Min improvement:   {results_df['improvement_pct'].min():+.1f}%")
print(f"  Max improvement:   {results_df['improvement_pct'].max():+.1f}%")

print("\n\nUtility Stats (Baseline):")
print(f"  Mean:  {results_df['baseline_utility'].mean():.4f}")
print(f"  Std:   {results_df['baseline_utility'].std():.4f}")
print(f"  Range: [{results_df['baseline_utility'].min():.4f}, {results_df['baseline_utility'].max():.4f}]")

print("\nUtility Stats (Tuned):")
print(f"  Mean:  {results_df['tuned_utility'].mean():.4f}")
print(f"  Std:   {results_df['tuned_utility'].std():.4f}")
print(f"  Range: [{results_df['tuned_utility'].min():.4f}, {results_df['tuned_utility'].max():.4f}]")

## 6. Save Results

In [ ]:
results_df.to_csv('utility_tuned_comparison.csv', index=False)
print("✓ Saved: utility_tuned_comparison.csv")

print("\n" + "="*70)
print("SUCCESS: Utility-tuned models tested")
print("="*70)
print(f"\nKey Finding:")
print(f"  Utility tuning improves average utility by {results_df['improvement_pct'].mean():.1f}%")
print(f"\nNext Steps:")
print(f"  1. Try other models: XGBoostUtilityTuned, GRUUtilityTuned")
print(f"  2. Tune hyperparameters: C, scale_pos_weight, max_depth")
print(f"  3. Use cross-validation for threshold selection")